# 09 — Preparación para espacio de datos

> **Catálogo homogéneo `showcase_master_catalog.json`** — mismos SKUs P1–P5 en 3D-BPP, Container Loading y Single Container. Cada tipo usa una **instancia showcase** compleja donde el benchmark **diferencia** los algoritmos del grupo.

Cada algoritmo implementado queda descrito como activo publicable con endpoint, esquemas de entrada/salida y tipos de problema soportados.


In [ ]:
# Configuración del entorno (ejecutar primero)
%matplotlib inline

import sys
from pathlib import Path
import matplotlib.pyplot as plt

_cwd = Path.cwd().resolve()
for _candidate in [_cwd, *_cwd.parents]:
    _nb = _candidate / "notebooks"
    if (_nb / "_shared" / "loaders.py").exists():
        if str(_nb) not in sys.path:
            sys.path.insert(0, str(_nb))
        break
else:
    raise RuntimeError("No se encontró notebooks/_shared. Abre el notebook desde packing-services/.")

from _shared.loaders import setup_paths
ROOT = setup_paths(_cwd)
from _shared import display, viz, runners

plt.rcParams.update({"figure.dpi": 110, "font.size": 11})
print(f"Proyecto: {ROOT}")


In [ ]:
from _shared.loaders import SHOWCASE_INSTANCES, BENCHMARK_GROUPS, SHOWCASE_DIFFERENTIATION, load_master_catalog
from packing_services.services.algorithm_input_service import AlgorithmInputService
catalog = runners.run_dataspace_catalog()
print('dataspace_ready:', catalog.dataspace_ready)
master = load_master_catalog()
print(f'Catálogo homogéneo: {master["catalog_version"]} — {master["description"]}')
import pandas as pd
from IPython.display import display as ipy_display
agg = [s for s in catalog.services if not s.service_name.endswith('Algorithm Service')]
rows = [{'Servicio': s.service_name, 'Problema': s.problem_type, 'Algoritmos': len(s.algorithms),
         'Input': s.input_schema, 'Output': s.output_schema, 'Endpoint': s.execution_endpoint} for s in agg]
ipy_display(pd.DataFrame(rows).style.hide(axis='index'))
algo_rows = [{'Algoritmo': s.algorithms[0], 'Input': s.input_schema, 'Output': s.output_schema,
              'Endpoint': s.execution_endpoint} for s in catalog.services if s.service_name.endswith('Algorithm Service')]
print(f'Servicios por algoritmo: {len(algo_rows)}')
ipy_display(pd.DataFrame(algo_rows).style.hide(axis='index'))
bench_rows = [{'Grupo': g, 'Instancia': SHOWCASE_INSTANCES.get(g, '—'), 'Diferenciación': SHOWCASE_DIFFERENTIATION.get(g, '—')} for g in BENCHMARK_GROUPS]
ipy_display(pd.DataFrame(bench_rows).style.hide(axis='index'))


In [ ]:
svc = AlgorithmInputService()
example = svc.build_input_example('multi_box_cartonization')
print('Ejemplo de entrada (CartonizationAlgorithmInput):', sorted(example.keys()))
result = runners.run_algorithm_execute('multi_box_cartonization', example)
display.show_execute_result(result)
print('Cajas usadas:', result.cartonization.selected_box_ids if result.cartonization else '—')


## Cierre del recorrido didáctico

1. **Cuatro tipos de problema** con instancia showcase dedicada (`examples/showcase_*_instance.json`)
2. **Veintinueve algoritmos** implementados + adaptador ``py3dbp`` (si está instalado)
3. **Nueve benchmarks** comparativos: 3D-BPP, híbrido, mejora 3D, metaheurísticas 3D, CL, Cartonization, SCL, Palletization, Stacking-aware
4. **Perfiles estándar** (`constructive`, `hybrid`, `box_selection`) para comparación reproducible
5. **Validación independiente** y **descriptores** listos para espacio de datos (`GET /api/v1/services`)
6. **Un endpoint por algoritmo** con salida `AlgorithmExecuteResponse` estandarizada

Cada benchmark usa un caso diseñado para **diferenciar** estrategias del mismo grupo — evidencia cuantitativa del avance del proyecto.
